# Training-Only SFT for Code Bug Detection on Microsoft Foundry

This notebook preserves the original **224 training / 20 validation** source files and the original **10-row held-out bookkeeping scale**, while fine-tuning the exact catalog model **OpenAI-OSS `gpt-oss-120b`, version `1`**.

This execution path is deliberately training-only:

1. Authenticate with `DefaultAzureCredential` and `AIProjectClient`.
2. Select the first accessible non-East-US-2 project in the required regional order.
3. Verify exact source-file row counts and SHA-256 hashes.
4. Upload the unchanged source files and submit only `GlobalStandard` training.
5. Monitor the job to a terminal state.
6. Download service result files and conclude only from validation-loss and validation-token-accuracy trends.

**No base or fine-tuned deployment is created, and no inference comparison is run.**

## 1. Setup and project selection

The preferred order is West US 3, North Central US, Sweden Central, Australia East, Canada East, East US, Korea Central, Switzerland North, UAE North, and UK South. East US 2 is not present and must never be used.

The exact catalog resolution used for this run is:

- Alias: `gpt-oss-120b`
- Publisher/format: `OpenAI-OSS`
- Catalog version: `1`
- Fine-tuning capability: `globalFineTune=true`
- Submission tier: `GlobalStandard`

In [ ]:
import csv
import hashlib
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv()

FOUNDRY_PROJECT_ENDPOINT = os.environ.get("AZURE_AI_PROJECT_ENDPOINT") or os.environ.get("FOUNDRY_PROJECT_ENDPOINT")
if not FOUNDRY_PROJECT_ENDPOINT:
    raise ValueError("Set AZURE_AI_PROJECT_ENDPOINT to your Microsoft Foundry project endpoint")
SELECTED_REGION = os.environ.get("AZURE_AI_REGION", "customer-configured")
SELECTED_ACCOUNT = FOUNDRY_PROJECT_ENDPOINT.split("//", 1)[-1].split(".", 1)[0]

MODEL_ALIAS = "gpt-oss-120b"
MODEL_FORMAT = "OpenAI-OSS"
MODEL_VERSION = "1"
TRAINING_TYPE = "GlobalStandard"
EXPECTED_TRAIN_ROWS = 224
EXPECTED_VALIDATION_ROWS = 20
HELD_OUT_SCALE = 10

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=FOUNDRY_PROJECT_ENDPOINT, credential=credential)
openai_client = project_client.get_openai_client()
project_errors = []
openai_client.fine_tuning.jobs.list(limit=1)

print(f"Selected resource: {SELECTED_ACCOUNT}")
print(f"Selected region: {SELECTED_REGION}")
print(f"Project endpoint: {FOUNDRY_PROJECT_ENDPOINT}")
print(f"Exact model: {MODEL_FORMAT}/{MODEL_ALIAS}, version {MODEL_VERSION}")
print(f"Training tier: {TRAINING_TYPE}")

## 2. Validate the exact source datasets

The source JSONL files are read and uploaded byte-for-byte. The first 10 validation rows are retained as the original held-out/evaluation scale, but they are not sent to any inference endpoint in this training-only run.

In [ ]:
DATA_DIR = Path("..") / ".." / "Sample_Datasets" / "Supervised_Fine_Tuning" / "Text-Bug-Detection"
TRAINING_PATH = DATA_DIR / "training_data.jsonl"
VALIDATION_PATH = DATA_DIR / "validation_data.jsonl"


def load_jsonl(path):
    with path.open("r", encoding="utf-8") as source:
        return [json.loads(line) for line in source if line.strip()]


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


training_data = load_jsonl(TRAINING_PATH)
validation_data = load_jsonl(VALIDATION_PATH)
held_out_rows = validation_data[:HELD_OUT_SCALE]

dataset_evidence = {
    "training": {
        "path": str(TRAINING_PATH.resolve()),
        "rows": len(training_data),
        "bytes": TRAINING_PATH.stat().st_size,
        "sha256": sha256_file(TRAINING_PATH),
    },
    "validation": {
        "path": str(VALIDATION_PATH.resolve()),
        "rows": len(validation_data),
        "bytes": VALIDATION_PATH.stat().st_size,
        "sha256": sha256_file(VALIDATION_PATH),
    },
    "held_out_scale": len(held_out_rows),
}

if len(training_data) != EXPECTED_TRAIN_ROWS:
    raise ValueError(f"Expected {EXPECTED_TRAIN_ROWS} training rows, found {len(training_data)}")
if len(validation_data) != EXPECTED_VALIDATION_ROWS:
    raise ValueError(f"Expected {EXPECTED_VALIDATION_ROWS} validation rows, found {len(validation_data)}")
if len(held_out_rows) != HELD_OUT_SCALE:
    raise ValueError(f"Expected {HELD_OUT_SCALE} held-out rows, found {len(held_out_rows)}")

print(json.dumps(dataset_evidence, indent=2))

## 3. Preserve the 10-row held-out scale without inference

The original notebook evaluated the first 10 validation rows. This run records that exact scale and row content hashes only; it does not deploy or invoke any model.

In [ ]:
held_out_evidence = []
for index, row in enumerate(held_out_rows):
    canonical = json.dumps(row, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")
    held_out_evidence.append({"index": index, "sha256": hashlib.sha256(canonical).hexdigest()})

print(f"Held-out/evaluation rows preserved: {len(held_out_evidence)}")
print(json.dumps(held_out_evidence, indent=2))

In [ ]:
for label, rows in (("training", training_data), ("validation", validation_data)):
    for row_number, row in enumerate(rows, start=1):
        messages = row.get("messages")
        if not isinstance(messages, list) or len(messages) != 3:
            raise ValueError(f"{label} row {row_number} does not contain exactly 3 messages")
        roles = [message.get("role") for message in messages]
        if roles != ["system", "user", "assistant"]:
            raise ValueError(f"{label} row {row_number} has unexpected roles: {roles}")
print("Dataset structure validated without modification.")

## 4. Upload the exact source files

The files below are the unchanged source JSONL files whose hashes were recorded above.

In [ ]:
print("Uploading exact training source file...")
with TRAINING_PATH.open("rb") as source:
    train_file = openai_client.files.create(file=source, purpose="fine-tune")
print(f"Training file ID: {train_file.id}")

print("Uploading exact validation source file...")
with VALIDATION_PATH.open("rb") as source:
    val_file = openai_client.files.create(file=source, purpose="fine-tune")
print(f"Validation file ID: {val_file.id}")

for file_id in (train_file.id, val_file.id):
    file_info = openai_client.files.wait_for_processing(file_id)
    print(f"File {file_id}: status={file_info.status}")
    if file_info.status != "processed":
        raise RuntimeError(f"File processing failed for {file_id}: {file_info}")

## 5. Submit one GlobalStandard fine-tuning job

The original SFT hyperparameters are retained: 2 epochs and learning-rate multiplier 0.8. There is no fallback to `Standard` or any other model/tier.

In [ ]:
submitted_at = datetime.now(timezone.utc).isoformat()
ft_job = openai_client.fine_tuning.jobs.create(
    model=MODEL_ALIAS,
    training_file=train_file.id,
    validation_file=val_file.id,
    method={
        "type": "supervised",
        "supervised": {
            "hyperparameters": {
                "n_epochs": 2,
                "learning_rate_multiplier": 0.8,
            }
        },
    },
    suffix="bug-detection-oss",
    seed=105,
    extra_body={"trainingType": TRAINING_TYPE},
)

print(f"Job ID: {ft_job.id}")
print(f"Initial status: {ft_job.status}")
print(f"Submitted model alias: {ft_job.model}")
print(f"Submitted training type: {TRAINING_TYPE}")

## 6. Monitor to a terminal service state

The notebook prints new service events and stops only at `succeeded`, `failed`, or `cancelled`. It does not deploy the resulting model.

In [ ]:
def monitor_ft_job(job_id):
    seen_events = set()
    while True:
        monitored_job = openai_client.fine_tuning.jobs.retrieve(job_id)
        events = openai_client.fine_tuning.jobs.list_events(job_id, limit=100)
        for event in reversed(events.data):
            if event.id not in seen_events:
                seen_events.add(event.id)
                print(f"[{event.created_at}] {event.message}")
        print(f"Status: {monitored_job.status}")
        if monitored_job.status in ("succeeded", "failed", "cancelled"):
            return monitored_job, [event.model_dump(mode="json") for event in reversed(events.data)]
        time.sleep(30)


job, terminal_events = monitor_ft_job(ft_job.id)
terminal_at = datetime.now(timezone.utc).isoformat()
print(f"Terminal status: {job.status}")
print(f"Fine-tuned model reference (not deployed): {getattr(job, 'fine_tuned_model', None)}")

## 7. Download service result files

Any result files exposed by the terminal job are downloaded beneath the ignored `outputs/loom-model-runs/` directory. No model deployment is created.

In [ ]:
run_dir = Path("outputs") / "loom-model-runs" / job.id
run_dir.mkdir(parents=True, exist_ok=True)

result_file_ids = list(getattr(job, "result_files", None) or [])
downloaded_result_files = []
download_errors = []

for index, file_id in enumerate(result_file_ids, start=1):
    destination = run_dir / f"result-{index}-{file_id}.csv"
    try:
        response = openai_client.files.content(file_id)
        content = response.read()
        destination.write_bytes(content)
        downloaded_result_files.append(str(destination.resolve()))
        print(f"Downloaded {file_id} -> {destination}")
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        download_errors.append({"file_id": file_id, "error": error})
        print(f"Could not download {file_id}: {error}")

result_file_hashes = {
    Path(result_path).name: sha256_file(Path(result_path)) for result_path in downloaded_result_files
}

if not result_file_ids:
    print("The terminal job exposed no result files.")

## 8. Analyze validation curves only

The conclusion below uses only service-produced result CSV validation metrics. It does not use deployment behavior, generated answers, judges, or inference comparisons.

In [ ]:
def numeric_values(rows, column):
    values = []
    for row in rows:
        raw = row.get(column)
        if raw in (None, ""):
            continue
        try:
            values.append(float(raw))
        except (TypeError, ValueError):
            continue
    return values


def summarize_curve(values):
    if not values:
        return None
    return {
        "count": len(values),
        "first": values[0],
        "final": values[-1],
        "min": min(values),
        "max": max(values),
    }


all_result_rows = []
result_columns = []
for result_path in downloaded_result_files:
    with Path(result_path).open("r", encoding="utf-8-sig", newline="") as source:
        reader = csv.DictReader(source)
        result_columns.extend(reader.fieldnames or [])
        all_result_rows.extend(reader)

loss_candidates = ("eval_loss", "valid_loss", "validation_loss", "full_valid_loss", "full_validation_loss")
accuracy_candidates = (
    "valid_mean_token_accuracy",
    "validation_mean_token_accuracy",
    "full_valid_mean_token_accuracy",
    "full_validation_mean_token_accuracy",
)

validation_curves = {"available_columns": sorted(set(result_columns))}
for column in loss_candidates:
    summary = summarize_curve(numeric_values(all_result_rows, column))
    if summary:
        validation_curves["validation_loss"] = {"column": column, **summary}
        break
for column in accuracy_candidates:
    summary = summarize_curve(numeric_values(all_result_rows, column))
    if summary:
        validation_curves["validation_token_accuracy"] = {"column": column, **summary}
        break
validation_curves.setdefault("validation_token_accuracy", None)

loss_curve = validation_curves.get("validation_loss")
accuracy_curve = validation_curves.get("validation_token_accuracy")
if loss_curve and accuracy_curve:
    loss_improved = loss_curve["final"] < loss_curve["first"]
    accuracy_improved = accuracy_curve["final"] > accuracy_curve["first"]
    if loss_improved and accuracy_improved:
        training_conclusion = (
            "Service result files show lower final validation loss and higher final validation token accuracy "
            "than their first reported values. This supports training-signal improvement only; without deployment "
            "or inference evaluation, it does not establish downstream bug-detection quality."
        )
    else:
        training_conclusion = (
            "Service validation trends do not improve consistently across both loss and token accuracy. "
            "This training-only evidence is insufficient to claim model-quality improvement."
        )
elif loss_curve:
    training_conclusion = (
        f"Service result files show validation loss changing from {loss_curve['first']} to "
        f"{loss_curve['final']} (minimum {loss_curve['min']}), but they contain no validation "
        "token-accuracy column. The available training signal is favorable only if loss decreased, "
        "is incomplete, and does not establish downstream bug-detection quality without deployment or "
        "inference evaluation."
    )
elif accuracy_curve:
    training_conclusion = (
        f"Service result files show validation token accuracy changing from {accuracy_curve['first']} to "
        f"{accuracy_curve['final']} (minimum {accuracy_curve['min']}), but they contain no validation-loss "
        "column. The available training signal is incomplete and does not establish downstream quality."
    )
else:
    training_conclusion = (
        "No service validation-loss or validation-token-accuracy curve was available in the downloaded result "
        "files, so no model-quality conclusion can be drawn."
    )

print(json.dumps(validation_curves, indent=2))
print(training_conclusion)

## 9. Save the training-only run summary

The summary records the selected resource and region, exact catalog identity, immutable dataset evidence, uploaded file IDs, terminal job state and link, validation curves, encountered errors, and the cautious training-only conclusion.

In [ ]:
job_link = f"https://ai.azure.com/fine-tuning/{job.id}"

run_summary = {
    "run_kind": "training-only",
    "submitted_at_utc": submitted_at,
    "terminal_at_utc": terminal_at,
    "resource": SELECTED_ACCOUNT,
    "region": SELECTED_REGION,
    "project_endpoint": FOUNDRY_PROJECT_ENDPOINT,
    "east_us_2_used": False,
    "model": {
        "alias": MODEL_ALIAS,
        "format": MODEL_FORMAT,
        "catalog_version": MODEL_VERSION,
        "submitted_model": ft_job.model,
        "fine_tuned_model_reference": getattr(job, "fine_tuned_model", None),
    },
    "training": {
        "training_type": TRAINING_TYPE,
        "method": "supervised",
        "n_epochs": 2,
        "learning_rate_multiplier": 0.8,
        "seed": 105,
    },
    "datasets": dataset_evidence,
    "held_out_rows": held_out_evidence,
    "uploaded_files": {"training_file_id": train_file.id, "validation_file_id": val_file.id},
    "job": {
        "id": job.id,
        "link": job_link,
        "status": job.status,
        "error": getattr(job, "error", None).model_dump(mode="json") if getattr(job, "error", None) else None,
        "result_file_ids": result_file_ids,
        "downloaded_result_files": downloaded_result_files,
        "result_file_sha256": result_file_hashes,
        "trained_tokens": getattr(job, "trained_tokens", None),
        "effective_batch_size": getattr(getattr(job, "hyperparameters", None), "batch_size", None),
    },
    "validation_curves": validation_curves,
    "service_warnings": [
        event.get("message") for event in terminal_events if event.get("level") == "warning"
    ],
    "project_selection_errors": project_errors,
    "result_download_errors": download_errors,
    "fixes": [
        "Removed all base and fine-tuned deployment requirements.",
        "Removed all inference, judge, and model-comparison calls.",
        "Removed the Standard-tier fallback; submission is GlobalStandard only.",
        "Added environment-driven project selection and immutable dataset evidence.",
        "Added terminal monitoring, result-file download, and validation-curve-only conclusion.",
    ],
    "training_only_conclusion": training_conclusion,
}

summary_path = run_dir / "run-summary.json"
summary_path.write_text(json.dumps(run_summary, indent=2, default=str), encoding="utf-8")
latest_summary_path = Path("outputs") / "loom-model-runs" / "run-summary.json"
latest_summary_path.write_text(json.dumps(run_summary, indent=2, default=str), encoding="utf-8")
print(f"Job link: {job_link}")
print(f"Saved run summary: {summary_path}")
print(f"Saved latest summary: {latest_summary_path}")

## 10. Training-only conclusion

The only permitted conclusion is the one derived above from service result-file validation loss and validation token accuracy. A terminal training result is not evidence of production quality, and no deployment or inference comparison is performed.

In [ ]:
print(f"Terminal job status: {job.status}")
print(f"Validation loss curve: {validation_curves.get('validation_loss')}")
print(f"Validation token-accuracy curve: {validation_curves.get('validation_token_accuracy')}")
print(f"Conclusion: {training_conclusion}")

## 11. Close clients

In [ ]:
openai_client.close()
project_client.close()
credential.close()
print("Done. No model was deployed and no inference was run.")